# Model Comparison

The pre-specified analysis of the simulation study for one arm: NEWS2 against a naive logistic regression, a GEE logistic model and gradient-boosted trees, with the H1, H2 and H3 decisions. The plan stays local (`plans/analysis-plan.md`) and is frozen by its SHA-256; this notebook refuses to read data when the plan changed.

Run it through `.venv/bin/python -m e2e.analysis --plan-sha256 <PLAN_SHA256>`, which executes it for the study and the null-control arm and writes the executed notebooks and the report under `artifacts/e2e/analysis/`. To open it in JupyterLab instead, start it from the repository root with `PYTHONPATH=$PWD` and set `ANALYSIS_SIGNAL`, `ANALYSIS_PLAN_SHA256` and `ANALYSIS_RESULTS` first. The data are synthetic; the results show how well each method recovers a planted signal, not clinical performance.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd

from jobs.analysis import data, run
from jobs.local_warehouse import load

ROOT = Path(os.environ.get("ANALYSIS_ROOT", Path.cwd().parent)).resolve()
SIGNAL = os.environ.get("ANALYSIS_SIGNAL", "study")
PLAN = ROOT / os.environ.get("ANALYSIS_PLAN", "plans/analysis-plan.md")
PLAN_SHA256 = data.check_plan(PLAN, os.environ["ANALYSIS_PLAN_SHA256"])
RESULTS = Path(os.environ["ANALYSIS_RESULTS"])
f"arm: {SIGNAL}; plan SHA-256 {PLAN_SHA256[:12]} matches the frozen hash"

In [ ]:
frames = data.load_frames(load.warehouse_schemas(SIGNAL))
results = run.analyze(frames)
pd.Series(results["analysis_set"], name="analysis set").to_frame()

## Discrimination (grouped 5-fold, out of fold)

In [ ]:
primary = results["primary"]
pd.DataFrame(
    {
        "AUC": primary["auc"],
        "95% CI low": {name: low for name, (low, _high) in primary["auc_interval"].items()},
        "95% CI high": {name: high for name, (_low, high) in primary["auc_interval"].items()},
    }
).round(3)

## H2: each model against NEWS2 (DeLong, Holm across three)

In [ ]:
pd.DataFrame(primary["comparisons"]).T.join(
    pd.DataFrame(primary["difference_interval"], index=["bootstrap difference low", "bootstrap difference high"]).T
).round(4)

## H1 and H3: GEE tests (clusters = split groups, bias-reduced covariance)

In [ ]:
tests = {"H1 vital signal": results["h1"], **{f"H3 {name}": test for name, test in results["h3"].items()}}
pd.DataFrame(tests).T[["statistic", "degrees_of_freedom", "p_value", "holm_p_value", "estimates"]]

## Decisions

In [ ]:
pd.Series(results["decisions"], name=SIGNAL).to_frame()

## Sensitivity Analyses

### 1. Encounter Clusters With the Usual Robust Covariance

In [ ]:
sensitivity = results["sensitivity"]
encounter_clusters = {"H1 vital signal": sensitivity["encounter_clusters"]["h1"]}
encounter_clusters |= {f"H3 {name}": test for name, test in sensitivity["encounter_clusters"]["h3"].items()}
pd.DataFrame(encounter_clusters).T[["statistic", "p_value", "holm_p_value"]]

### 2. Bootstrap Intervals of Each AUC Difference

Shown with H2 above (`bootstrap difference low` and `high`).

### 3. Fixed Split, Test Encounters

In [ ]:
fixed = sensitivity["fixed_split"]
pd.DataFrame({"AUC": fixed["auc"]}).join(pd.DataFrame(fixed["comparisons"]).T).round(4)

### 4. Heart-Rate Rise by Moderator (Calibration Formulation)

In [ ]:
pd.DataFrame(sensitivity["heart_rate_rise"]).T[["statistic", "p_value", "holm_p_value", "estimates"]]

## Known Truth

The calibration seed's values recorded in `config/planted_signal.json`, which the study is expected to land near by sampling variation.

In [ ]:
calibration = json.loads((ROOT / "config" / "planted_signal.json").read_text(encoding="utf-8"))["calibration"]
pd.DataFrame({"calibration seed": calibration["achieved"], "null control on that seed": calibration["null_control"]}).T

In [ ]:
RESULTS.parent.mkdir(parents=True, exist_ok=True)
RESULTS.write_text(json.dumps({"signal": SIGNAL, "plan_sha256": PLAN_SHA256, **results}, indent=2, sort_keys=True) + "\n", encoding="utf-8")
RESULTS.name